# Coleta dos dados da API

In [515]:
import requests
import json

import pandas as pd
import math

## Funções para Query

Aqui, definimos as funções para, efetivamente, fazer as requisições para a API da AniList.

Começamos definindo uma constante e uma função para facilitar a escrita das requisições.

In [516]:
URL = 'https://graphql.anilist.co'

In [517]:
def embedPageQuery(query:str):
    """Coloca uma query dentro da estrutura de requisição por páginas da AniList.
        Isso evita a reescrita do mesmo código"""
    
    embededQuery =  '''
                    query ($page: Int, $perPage: Int) {
                        page: Page (page: $page, perPage: $perPage){
                    '''
    embededQuery += query
    embededQuery += "}}"
    
    return embededQuery

Agora, vamos definiar as funções para solicitar os dados à API.

`queryPages` é a função onde efetivamente é realizada a requisição. A AniList estrurua a requisição de vários dados através de "páginas" (_pages_), as quais tem no máximo 50 entradas. Portanto, se quisermos mais, vamos ter que fazer várias requisições. Portanto, esse código faz várias requisições e concatena as respostas.

O `getPages`, por outro lado, considerada que os dados da AniList vão mudar muito pouco com o tempo, então para não precisar fazer a requisição toda vez que executar o código, os dados recebidos por `queryPages` são salvos no arquivo .json, caso ele ainda não exista. No caso do arquivo já existir, o código pega os dados diretos desse arquivos, sem nem fazer a requisação à API.

In [518]:
def queryPages(url:str, query:str, total:int=50):
    """Coleta <total> entradas da query <query> em <url>"""
    
    lasting = total
    
    pages = []
    while lasting > 0:

        variables = {
            'page': math.floor((total-lasting)/50)+1,
            'perPage': lasting
        }

        print("Requisitando dados...")
        response = requests.post(url, json={'query': embedPageQuery(query), 'variables': variables})
        pages.append(json.loads(response.content)["data"]['page'])

        lasting -= 50

    jointPages = []
    key = list(pages[0].keys())[0]
    for page in pages:
        jointPages += page[key]

    return jointPages

In [519]:
def getPages(url:str, query:str, name:str, total:int=50):
    """Busca pela query nos arquivos de cache. Se não encontrar, faz a requisição."""
    
    try:
        with open(f'queryCache/{name}.json', 'r') as file:
            data = json.load(file)
            
        if data['query'] != query or data['entryAmount'] != total:
            raise FileNotFoundError # Não é exatamente '_file_ not found', mas é perto o suficiente 
        
        pages = data['pages']
        print("Dados em cache!")

    except FileNotFoundError:
        pages = queryPages(url, query, total)

        data = {
            'pages': pages,
            'query': query,
            'entryAmount': total
        }

        with open(f'queryCache/{name}.json', 'w') as file:
            file.write(json.dumps(data, indent=4))

    return pages

## Requisições

Definimos as queries a ser feitas

In [520]:
animeQuery = '''media (sort: SCORE_DESC, type: ANIME) {
                    id
                    title {
                        romaji
                        english
                        native
                    }
                    coverImage {
                        large
                    }
                    type
                    format
                    genres
                    status
                    description
                    startDate {
                        year
                        month
                        day
                    }
                    endDate {
                        year
                        month
                        day
                    }
                    episodes
                }
            '''

In [521]:
mangaQuery = '''media (sort: SCORE_DESC, type: MANGA) {
                    id
                    title {
                        romaji
                        english
                        native
                    }
                    coverImage {
                        large
                    }
                    type
                    format
                    genres
                    status
                    description
                    startDate {
                        year
                        month
                        day
                    }
                    endDate {
                        year
                        month
                        day
                    }
                    chapters
                    volumes
                }
            '''

In [522]:
studioQuery =   '''studios (sort: FAVOURITES_DESC) {
                        id
                        name
                    }
                '''

In [523]:
charQuery = '''characters  (sort: FAVOURITES_DESC) {
                    id
                    name {
                        full
                    }
                    image {
                        large
                    }
                    description
                    gender
                    age
                }
            '''

In [524]:
staffQuery = '''staff (sort: FAVOURITES_DESC) {
                    id
                    image {
                        large
                    }
                    name {
                        full
                    }
                    description
                    gender
                    age
                    primaryOccupations
                }
            '''

In [525]:
vaQuery = '''staff (sort: FAVOURITES_DESC, primaryOccupations: 'Voice Actor') {
                    id
                    image {
                        large
                    }
                    name {
                        full
                    }
                    description
                    gender
                    age
                    primaryOccupations
                    languageV2
                }
            '''

In [526]:
userQuery = '''users (sort: WATCHED_TIME_DESC){
                    id
                    name
                    avatar {
                        large
                    }
                    createdAt
                }
            '''

Organizamos as queries a serem feitas junto ao nome do seu arquivo e a quantidade de elementos para aquela query

In [ ]:
# variables = {
#             'page': 1,
#             'perPage': 50
#         }
# a = requests.post(URL, json={'query': embedPageQuery(vaQuery), 'variables': variables})
# json.loads(a.content)

{'errors': [{'message': 'The AniList API has been temporarily disabled due to severe stability issues.',
   'status': 403,
   'locations': [{'line': 1, 'column': 1}]}],
 'data': None}

In [528]:
queries = {
    'anime':    (animeQuery,    250),
    'manga':    (mangaQuery,    250),
    'studio':   (studioQuery,   50),
    'character':(charQuery,     250),
    'staff':    (staffQuery,    150),
    # 'va':       (vaQuery,       100),
    'user':     (userQuery,     80)
}

Fazermos a requisição

In [529]:
dfs = {}
for name, (query, total) in queries.items():
    print(f"\n-- {name.capitalize()} --")
    pages = getPages(URL, query, name, total)
    dfs[name] = pd.DataFrame(pages)


-- Anime --
Dados em cache!

-- Manga --
Dados em cache!

-- Studio --
Dados em cache!

-- Character --
Dados em cache!

-- Staff --
Dados em cache!

-- User --
Dados em cache!


## Retorno das queries

In [530]:
dfs['anime']

,id,title,coverImage,type,format,genres,status,description,startDate,endDate,episodes
0,154587,"{'romaji': 'Sousou no Frieren', 'english': 'Fr...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Adventure, Drama, Fantasy]",FINISHED,The adventure is over but life goes on for an ...,"{'year': 2023, 'month': 9, 'day': 29}","{'year': 2024, 'month': 3, 'day': 22}",28.0
1,114129,"{'romaji': 'Gintama: THE FINAL', 'english': 'G...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Action, Comedy, Drama, Sci-Fi]",FINISHED,Two years have passed following the Tendoshuu'...,"{'year': 2021, 'month': 1, 'day': 8}","{'year': 2021, 'month': 1, 'day': 8}",1.0
2,20996,"{'romaji': 'Gintama°', 'english': 'Gintama Sea...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Comedy, Drama, Sci-Fi]",FINISHED,"Gintoki, Shinpachi, and Kagura return as the f...","{'year': 2015, 'month': 4, 'day': 8}","{'year': 2016, 'month': 3, 'day': 30}",51.0
3,189046,{'romaji': 'Re:Zero kara Hajimeru Isekai Seika...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Drama, Fantasy, Psychologi...",FINISHED,"In the deadly battle at Priestella, Subaru and...","{'year': 2026, 'month': 4, 'day': 8}","{'year': 2026, 'month': 9, 'day': 30}",19.0
4,171627,"{'romaji': 'Chainsaw Man: Reze-hen', 'english'...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Action, Drama, Horror, Romance, Supernatural]",FINISHED,"Denji became “Chainsaw Man”, a boy with a devi...","{'year': 2025, 'month': 9, 'day': 19}","{'year': 2025, 'month': 9, 'day': 19}",1.0
...,...,...,...,...,...,...,...,...,...,...,...
245,101903,"{'romaji': 'Kaze ga Tsuyoku Fuiteiru', 'englis...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Drama, Sports]",FINISHED,"Former ace runner of Sendai Josei High School,...","{'year': 2018, 'month': 10, 'day': 3}","{'year': 2019, 'month': 3, 'day': 27}",23.0
246,108632,{'romaji': 'Re:Zero kara Hajimeru Isekai Seika...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Drama, Fantasy, Psychologi...",FINISHED,"Even after dying countless times, Subaru final...","{'year': 2020, 'month': 7, 'day': 8}","{'year': 2020, 'month': 9, 'day': 30}",13.0
247,109403,"{'romaji': 'Non Non Biyori: Nonstop', 'english...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Comedy, Slice of Life]",FINISHED,Fresh excitement brews as someone new moves in...,"{'year': 2021, 'month': 1, 'day': 11}","{'year': 2021, 'month': 3, 'day': 29}",12.0
248,140960,"{'romaji': 'SPY×FAMILY', 'english': 'SPY x FAM...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Comedy, Slice of Life, Supernatural]",FINISHED,Everyone has a part of themselves they cannot ...,"{'year': 2022, 'month': 4, 'day': 9}","{'year': 2022, 'month': 6, 'day': 25}",12.0


In [531]:
dfs['manga']

,id,title,coverImage,type,format,genres,status,description,startDate,endDate,chapters,volumes
0,30002,"{'romaji': 'Berserk', 'english': 'Berserk', 'n...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Drama, Fantasy, Horror, Ps...",RELEASING,"His name is Guts, the Black Swordsman, a feare...","{'year': 1989, 'month': 8, 'day': 25}","{'year': None, 'month': None, 'day': None}",NaN,NaN
1,31706,{'romaji': 'JoJo no Kimyou na Bouken: Steel Ba...,{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Drama, Mystery, Supernatural]",FINISHED,Riders from around the world gather in the Wil...,"{'year': 2004, 'month': 1, 'day': 19}","{'year': 2011, 'month': 4, 'day': 19}",95.0,24.0
2,30656,"{'romaji': 'Vagabond', 'english': 'Vagabond', ...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Drama]",RELEASING,"At seventeen years of age, Miyamoto Musashi--s...","{'year': 1998, 'month': 9, 'day': 3}","{'year': None, 'month': None, 'day': None}",NaN,NaN
3,30013,"{'romaji': 'ONE PIECE', 'english': 'One Piece'...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Comedy, Fantasy]",RELEASING,"As a child, Monkey D. Luffy was inspired to be...","{'year': 1997, 'month': 7, 'day': 22}","{'year': None, 'month': None, 'day': None}",NaN,NaN
4,64053,{'romaji': 'Umineko no Naku Koro ni Chiru Epis...,{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Drama, Fantasy, Mystery, Supernatural]",FINISHED,It's time once again for the annual family gat...,"{'year': 2012, 'month': 1, 'day': 21}","{'year': 2015, 'month': 6, 'day': 22}",42.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...
245,113891,"{'romaji': 'Souyaku Toaru Majutsu no Index', '...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,NOVEL,"[Action, Comedy, Fantasy, Sci-Fi]",RELEASING,Science and magic exist side by side in this w...,"{'year': 2020, 'month': 2, 'day': 7}","{'year': None, 'month': None, 'day': None}",NaN,NaN
246,143749,"{'romaji': 'Zoku Owarimonogatari', 'english': ...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,NOVEL,"[Comedy, Drama, Mystery, Psychological, Supern...",FINISHED,Just when we thought the darkness menacing the...,"{'year': 2014, 'month': 9, 'day': 17}","{'year': 2014, 'month': 9, 'day': 17}",33.0,1.0
247,30010,"{'romaji': 'xxxHOLiC', 'english': 'xxxHOLiC', ...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Comedy, Drama, Mystery, Supernatural]",FINISHED,Kimihiro Watanuki is haunted by visions of gho...,"{'year': 2003, 'month': 2, 'day': 24}","{'year': 2011, 'month': 2, 'day': 9}",214.0,19.0
248,30704,"{'romaji': 'Trigun Maximum', 'english': 'Trigu...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Comedy, Drama, Sci-Fi]",FINISHED,"Much of the damage attributed to ""Vash"" is cau...","{'year': 1997, 'month': 10, 'day': 31}","{'year': 2007, 'month': 3, 'day': 30}",104.0,14.0


In [532]:
dfs['studio']

,id,name
0,2,Kyoto Animation
1,11,MADHOUSE
2,569,MAPPA
3,43,ufotable
4,4,bones
5,21,Studio Ghibli
6,858,WIT STUDIO
7,561,A-1 Pictures
8,44,Shaft
9,803,TRIGGER


In [533]:
dfs['character']

,id,name,image,description,gender,age
0,127691,{'full': 'Satoru Gojou'},{'large': 'https://s4.anilist.co/file/anilistc...,"__Height:__ 190 cm (6'3"")\n__Occupation:__ Juj...",Male,28
1,45627,{'full': 'Levi'},{'large': 'https://s4.anilist.co/file/anilistc...,"__Height:__ 160 cm (5'3"")\n\nSquad Captain of ...",Male,NaN
2,40,{'full': 'Luffy Monkey'},{'large': 'https://s4.anilist.co/file/anilistc...,__Height:__ 172 cm\n__Affiliations:__ ~!Straw ...,Male,17-
3,27,{'full': 'Killua Zoldyck'},{'large': 'https://s4.anilist.co/file/anilistc...,"__Height:__ 158 cm (5'2"")\n\nKillua is a membe...",Male,11-
4,40882,{'full': 'Eren Yeager'},{'large': 'https://s4.anilist.co/file/anilistc...,"__Height:__ 170 cm (5'7"") - 183 cm (6'0"")\n__A...",Male,15-
...,...,...,...,...,...,...
245,157116,{'full': 'Chousou'},{'large': 'https://s4.anilist.co/file/anilistc...,Chousou is half human and half cursed spirit a...,Male,150+
246,46498,{'full': 'Jean Kirstein'},{'large': 'https://s4.anilist.co/file/anilistc...,"__Height__: 175 cm (5'8"")\n__Relatives__: [Mot...",Male,16-
247,67069,{'full': 'Yui Yuigahama'},{'large': 'https://s4.anilist.co/file/anilistc...,"Yui is a cheerful, outgoing girl who becomes t...",Female,16-17
248,19565,{'full': 'Yui Hirasawa'},{'large': 'https://s4.anilist.co/file/anilistc...,__Height:__ 156 cm\n\nYui Hirasawa is one of t...,Female,15-18


In [534]:
dfs['staff']

,id,image,name,description,gender,age,primaryOccupations
0,95591,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kenjirou Tsuda'},"__Height:__ 170 cm (5'7"")\n__Agency:__ Amuleto...",Male,55.0,[Voice Actor]
1,95118,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Hiroshi Kamiya'},"**Height:** 167 cm (5'6"")\n\n[Agency Profile (...",Male,51.0,[Voice Actor]
2,95869,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Saori Hayami'},__Agency:__ I'm Enterprise\n\n[Profile](https:...,Female,35.0,"[Voice Actor, Vocalist]"
3,95185,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kana Hanazawa'},"__Height:__ 156 cm (5'1"")\n\n[Profile](https:/...",Female,37.0,"[Voice Actor, Vocalist]"
4,95065,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Mamoru Miyano'},"__Height:__ 182 cm (6'0"")\n\n[Website](https:/...",Male,43.0,[Voice Actor]
...,...,...,...,...,...,...,...
145,116971,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Daiki Yamashita'},[Profile (Arts Vision)](https://www.artsvision...,Male,37.0,[Voice Actor]
146,133507,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Chiaki Kobayashi'},**Agency:** Office Osawa\n\n[Twitter](https://...,Male,32.0,[Voice Actor]
147,119976,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'MYTH & ROID'},MYTH & ROID is a musical unit consisting of mu...,NaN,NaN,[Band]
148,119416,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kaede Hondo'},__Agency:__ I'm Enterprise\n\n[Agency Profile]...,Female,30.0,[Voice Actor]


In [535]:
dfs['user']

,id,name,avatar,createdAt
0,219847,Mand3012,{'large': 'https://s4.anilist.co/file/anilistc...,1.546116e+09
1,61155,7elou,{'large': 'https://s4.anilist.co/file/anilistc...,1.451297e+09
2,63191,killkill,{'large': 'https://s4.anilist.co/file/anilistc...,1.453377e+09
3,76074,MaherAnime,{'large': 'https://s4.anilist.co/file/anilistc...,1.468717e+09
4,34111,Ichigokun,{'large': 'https://s4.anilist.co/file/anilistc...,1.415714e+09
...,...,...,...,...
75,35971,ANON192,{'large': 'https://s4.anilist.co/file/anilistc...,1.418461e+09
76,37820,BlackOcean,{'large': 'https://s4.anilist.co/file/anilistc...,1.420436e+09
77,48303,Enndo,{'large': 'https://s4.anilist.co/file/anilistc...,1.435123e+09
78,76717,qubiq,{'large': 'https://s4.anilist.co/file/anilistc...,1.469833e+09


## Expansão de colunas compostas

Na resposta da API, muitas colunas são compostas, sendo seus valores listas ou dicionários. Vamos expandí-las em tabelas próprias de acordo com o modelo lógico.

### Títulos

In [536]:
dfs['title'] = pd.DataFrame()

animeTitles = pd.DataFrame()
mangaTitles = pd.DataFrame()

animeTitles['id_midia'] = dfs['anime']['id']
mangaTitles['id_midia'] = dfs['anime']['id']

titleTypes = ['romaji', 'english', 'native']

for titleType in titleTypes:
    animeTitles[titleType] = dfs['anime']['title'].apply(lambda x: x[titleType])
    mangaTitles[titleType] = dfs['manga']['title'].apply(lambda x: x[titleType])


dfs['title'] = pd.concat([animeTitles, mangaTitles])

dfs['title']

,id_midia,romaji,english,native
0,154587,Sousou no Frieren,Frieren: Beyond Journey’s End,葬送のフリーレン
1,114129,Gintama: THE FINAL,Gintama: THE VERY FINAL,銀魂 THE FINAL
2,20996,Gintama°,Gintama Season 3,銀魂゜
3,189046,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,Re:ZERO -Starting Life in Another World- Season 4,Re:ゼロから始める異世界生活 4th season
4,171627,Chainsaw Man: Reze-hen,Chainsaw Man – The Movie: Reze Arc,チェンソーマン レゼ篇
...,...,...,...,...
245,101903,Souyaku Toaru Majutsu no Index,NaN,創約とある魔術の禁書目録
246,108632,Zoku Owarimonogatari,Zoku Owarimonogatari: End Tale (Cont.),続終物語
247,109403,xxxHOLiC,xxxHOLiC,xxxHOLiC
248,140960,Trigun Maximum,Trigun Maximum,トライガン・マキシマム


In [537]:
del dfs['anime']['title']
del dfs['manga']['title']

### Gêneros

In [538]:
compondGenres = pd.concat([dfs['anime'][['id', 'genres']], dfs['manga'][['id', 'genres']]])

dfs['genre'] = pd.DataFrame(columns=['id_midia', 'name'])

# bad and stinky code but god has punished me for my hubris and my work is never finished 
# so I don't have the time to fix it
for id, occupations in compondGenres.values:
    for genre in occupations:
        dfs['genre'].loc[len(dfs['genre'])] = [id, genre]

dfs['genre']

,id_midia,name
0,154587,Adventure
1,154587,Drama
2,154587,Fantasy
3,114129,Action
4,114129,Comedy
...,...,...
1847,30704,Sci-Fi
1848,30869,Action
1849,30869,Adventure
1850,30869,Comedy


In [539]:
del dfs['anime']['genres']
del dfs['manga']['genres']

### Ocupações

In [540]:
compondOccupations = dfs['staff'][['id', 'primaryOccupations']]

dfs['occupation'] = pd.DataFrame(columns=['id_staff', 'name'])

# bad and stinky code but god has punished me for my hubris and my work is never finished 
# so I don't have the time to fix it
for id, occupations in compondOccupations.values:
    for occupation in occupations:
        dfs['occupation'].loc[len(dfs['occupation'])] = [id, occupation]

dfs['occupation']

,id_staff,name
0,95591,Voice Actor
1,95118,Voice Actor
2,95869,Voice Actor
3,95869,Vocalist
4,95185,Voice Actor
...,...,...
196,133507,Voice Actor
197,119976,Band
198,119416,Voice Actor
199,95027,Voice Actor


In [541]:
del dfs['staff']['primaryOccupations']

### Campos atômicos dentro de listas e dicionários

#### Imagens

In [542]:
extractImage = lambda x: x['large']

dfs['anime']['coverImage'] = dfs['anime']['coverImage'].apply(extractImage)
dfs['manga']['coverImage'] = dfs['manga']['coverImage'].apply(extractImage)
dfs['staff']['image'] = dfs['staff']['image'].apply(extractImage)
dfs['user']['avatar'] = dfs['user']['avatar'].apply(extractImage)
dfs['character']['image'] = dfs['character']['image'].apply(extractImage)

In [543]:
dfs['character']['image']

0      https://s4.anilist.co/file/anilistcdn/characte...
1      https://s4.anilist.co/file/anilistcdn/characte...
2      https://s4.anilist.co/file/anilistcdn/characte...
3      https://s4.anilist.co/file/anilistcdn/characte...
4      https://s4.anilist.co/file/anilistcdn/characte...
                             ...                        
245    https://s4.anilist.co/file/anilistcdn/characte...
246    https://s4.anilist.co/file/anilistcdn/characte...
247    https://s4.anilist.co/file/anilistcdn/characte...
248    https://s4.anilist.co/file/anilistcdn/characte...
249    https://s4.anilist.co/file/anilistcdn/characte...
Name: image, Length: 250, dtype: str

#### Nomes

In [544]:
extractName = lambda x: x['full']

dfs['character']['name'] = dfs['character']['name'].apply(extractName)
dfs['staff']['name'] = dfs['staff']['name'].apply(extractName)

In [545]:
dfs['staff']['name']

0        Kenjirou Tsuda
1        Hiroshi Kamiya
2          Saori Hayami
3         Kana Hanazawa
4         Mamoru Miyano
             ...       
145     Daiki Yamashita
146    Chiaki Kobayashi
147         MYTH & ROID
148         Kaede Hondo
149       Yukari Tamura
Name: name, Length: 150, dtype: str

## Tipagem

Alguns tipos vieram inapropriados, como float quando deveria ser inteiro ou datas quebradas de formas estranhas. Vamos ajeitas isso!

### Datas

In [546]:
def dict2timestamp(x):
    try:
        return pd.Timestamp(year=x['year'], month=x['month'], day=x['day'])
    except:
        return None

dfs['anime']['startDate'] = dfs['anime']['startDate'].apply(dict2timestamp)
dfs['anime']['endDate']   = dfs['anime']['endDate'].apply(dict2timestamp)

dfs['manga']['startDate'] = dfs['manga']['startDate'].apply(dict2timestamp)
dfs['manga']['endDate']   = dfs['manga']['endDate'].apply(dict2timestamp)

In [547]:
dfs['anime']['startDate']

0     2023-09-29
1     2021-01-08
2     2015-04-08
3     2026-04-08
4     2025-09-19
         ...    
245   2018-10-03
246   2020-07-08
247   2021-01-11
248   2022-04-09
249   2023-12-01
Name: startDate, Length: 250, dtype: datetime64[us]

In [548]:
dfs['user']['createdAt'] = pd.to_datetime(dfs['user']['createdAt'], unit='s')

In [549]:
dfs['user']['createdAt']

0    2018-12-29 20:33:13
1    2015-12-28 10:10:56
2    2016-01-21 11:46:01
3    2016-07-17 00:49:56
4    2014-11-11 13:47:18
             ...        
75   2014-12-13 08:52:45
76   2015-01-05 05:32:55
77   2015-06-24 05:14:19
78   2016-07-29 22:52:13
79   2018-09-12 12:45:29
Name: createdAt, Length: 80, dtype: datetime64[s]

### Numeros

In [550]:
dfs['anime']['episodes'] = dfs['anime']['episodes'].astype('Int64')

In [551]:
dfs['manga']['chapters'] = dfs['manga']['chapters'].astype('Int64')
dfs['manga']['volumes'] = dfs['manga']['chapters'].astype('Int64')

In [552]:
dfs['staff']['age'] = dfs['staff']['age'].astype('Int64')

## Dados limpos

In [553]:
dfs['anime']

,id,coverImage,type,format,status,description,startDate,endDate,episodes
0,154587,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,The adventure is over but life goes on for an ...,2023-09-29,2024-03-22,28
1,114129,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,MOVIE,FINISHED,Two years have passed following the Tendoshuu'...,2021-01-08,2021-01-08,1
2,20996,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"Gintoki, Shinpachi, and Kagura return as the f...",2015-04-08,2016-03-30,51
3,189046,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"In the deadly battle at Priestella, Subaru and...",2026-04-08,2026-09-30,19
4,171627,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,MOVIE,FINISHED,"Denji became “Chainsaw Man”, a boy with a devi...",2025-09-19,2025-09-19,1
...,...,...,...,...,...,...,...,...,...
245,101903,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"Former ace runner of Sendai Josei High School,...",2018-10-03,2019-03-27,23
246,108632,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"Even after dying countless times, Subaru final...",2020-07-08,2020-09-30,13
247,109403,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,Fresh excitement brews as someone new moves in...,2021-01-11,2021-03-29,12
248,140960,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,Everyone has a part of themselves they cannot ...,2022-04-09,2022-06-25,12


In [554]:
dfs['manga']

,id,coverImage,type,format,status,description,startDate,endDate,chapters,volumes
0,30002,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,RELEASING,"His name is Guts, the Black Swordsman, a feare...",1989-08-25,NaT,<NA>,<NA>
1,31706,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,Riders from around the world gather in the Wil...,2004-01-19,2011-04-19,95,95
2,30656,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,RELEASING,"At seventeen years of age, Miyamoto Musashi--s...",1998-09-03,NaT,<NA>,<NA>
3,30013,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,RELEASING,"As a child, Monkey D. Luffy was inspired to be...",1997-07-22,NaT,<NA>,<NA>
4,64053,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,It's time once again for the annual family gat...,2012-01-21,2015-06-22,42,42
...,...,...,...,...,...,...,...,...,...,...
245,113891,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,NOVEL,RELEASING,Science and magic exist side by side in this w...,2020-02-07,NaT,<NA>,<NA>
246,143749,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,NOVEL,FINISHED,Just when we thought the darkness menacing the...,2014-09-17,2014-09-17,33,33
247,30010,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,Kimihiro Watanuki is haunted by visions of gho...,2003-02-24,2011-02-09,214,214
248,30704,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,"Much of the damage attributed to ""Vash"" is cau...",1997-10-31,2007-03-30,104,104


In [555]:
dfs['studio']

,id,name
0,2,Kyoto Animation
1,11,MADHOUSE
2,569,MAPPA
3,43,ufotable
4,4,bones
5,21,Studio Ghibli
6,858,WIT STUDIO
7,561,A-1 Pictures
8,44,Shaft
9,803,TRIGGER


In [556]:
dfs['character']

,id,name,image,description,gender,age
0,127691,Satoru Gojou,https://s4.anilist.co/file/anilistcdn/characte...,"__Height:__ 190 cm (6'3"")\n__Occupation:__ Juj...",Male,28
1,45627,Levi,https://s4.anilist.co/file/anilistcdn/characte...,"__Height:__ 160 cm (5'3"")\n\nSquad Captain of ...",Male,NaN
2,40,Luffy Monkey,https://s4.anilist.co/file/anilistcdn/characte...,__Height:__ 172 cm\n__Affiliations:__ ~!Straw ...,Male,17-
3,27,Killua Zoldyck,https://s4.anilist.co/file/anilistcdn/characte...,"__Height:__ 158 cm (5'2"")\n\nKillua is a membe...",Male,11-
4,40882,Eren Yeager,https://s4.anilist.co/file/anilistcdn/characte...,"__Height:__ 170 cm (5'7"") - 183 cm (6'0"")\n__A...",Male,15-
...,...,...,...,...,...,...
245,157116,Chousou,https://s4.anilist.co/file/anilistcdn/characte...,Chousou is half human and half cursed spirit a...,Male,150+
246,46498,Jean Kirstein,https://s4.anilist.co/file/anilistcdn/characte...,"__Height__: 175 cm (5'8"")\n__Relatives__: [Mot...",Male,16-
247,67069,Yui Yuigahama,https://s4.anilist.co/file/anilistcdn/characte...,"Yui is a cheerful, outgoing girl who becomes t...",Female,16-17
248,19565,Yui Hirasawa,https://s4.anilist.co/file/anilistcdn/characte...,__Height:__ 156 cm\n\nYui Hirasawa is one of t...,Female,15-18


In [557]:
dfs['staff']

,id,image,name,description,gender,age
0,95591,https://s4.anilist.co/file/anilistcdn/staff/la...,Kenjirou Tsuda,"__Height:__ 170 cm (5'7"")\n__Agency:__ Amuleto...",Male,55
1,95118,https://s4.anilist.co/file/anilistcdn/staff/la...,Hiroshi Kamiya,"**Height:** 167 cm (5'6"")\n\n[Agency Profile (...",Male,51
2,95869,https://s4.anilist.co/file/anilistcdn/staff/la...,Saori Hayami,__Agency:__ I'm Enterprise\n\n[Profile](https:...,Female,35
3,95185,https://s4.anilist.co/file/anilistcdn/staff/la...,Kana Hanazawa,"__Height:__ 156 cm (5'1"")\n\n[Profile](https:/...",Female,37
4,95065,https://s4.anilist.co/file/anilistcdn/staff/la...,Mamoru Miyano,"__Height:__ 182 cm (6'0"")\n\n[Website](https:/...",Male,43
...,...,...,...,...,...,...
145,116971,https://s4.anilist.co/file/anilistcdn/staff/la...,Daiki Yamashita,[Profile (Arts Vision)](https://www.artsvision...,Male,37
146,133507,https://s4.anilist.co/file/anilistcdn/staff/la...,Chiaki Kobayashi,**Agency:** Office Osawa\n\n[Twitter](https://...,Male,32
147,119976,https://s4.anilist.co/file/anilistcdn/staff/la...,MYTH & ROID,MYTH & ROID is a musical unit consisting of mu...,NaN,<NA>
148,119416,https://s4.anilist.co/file/anilistcdn/staff/la...,Kaede Hondo,__Agency:__ I'm Enterprise\n\n[Agency Profile]...,Female,30


In [558]:
dfs['user']

,id,name,avatar,createdAt
0,219847,Mand3012,https://s4.anilist.co/file/anilistcdn/user/ava...,2018-12-29 20:33:13
1,61155,7elou,https://s4.anilist.co/file/anilistcdn/user/ava...,2015-12-28 10:10:56
2,63191,killkill,https://s4.anilist.co/file/anilistcdn/user/ava...,2016-01-21 11:46:01
3,76074,MaherAnime,https://s4.anilist.co/file/anilistcdn/user/ava...,2016-07-17 00:49:56
4,34111,Ichigokun,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-11-11 13:47:18
...,...,...,...,...
75,35971,ANON192,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-12-13 08:52:45
76,37820,BlackOcean,https://s4.anilist.co/file/anilistcdn/user/ava...,2015-01-05 05:32:55
77,48303,Enndo,https://s4.anilist.co/file/anilistcdn/user/ava...,2015-06-24 05:14:19
78,76717,qubiq,https://s4.anilist.co/file/anilistcdn/user/ava...,2016-07-29 22:52:13


In [559]:
dfs['title']

,id_midia,romaji,english,native
0,154587,Sousou no Frieren,Frieren: Beyond Journey’s End,葬送のフリーレン
1,114129,Gintama: THE FINAL,Gintama: THE VERY FINAL,銀魂 THE FINAL
2,20996,Gintama°,Gintama Season 3,銀魂゜
3,189046,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,Re:ZERO -Starting Life in Another World- Season 4,Re:ゼロから始める異世界生活 4th season
4,171627,Chainsaw Man: Reze-hen,Chainsaw Man – The Movie: Reze Arc,チェンソーマン レゼ篇
...,...,...,...,...
245,101903,Souyaku Toaru Majutsu no Index,NaN,創約とある魔術の禁書目録
246,108632,Zoku Owarimonogatari,Zoku Owarimonogatari: End Tale (Cont.),続終物語
247,109403,xxxHOLiC,xxxHOLiC,xxxHOLiC
248,140960,Trigun Maximum,Trigun Maximum,トライガン・マキシマム


In [560]:
dfs['staff']

,id,image,name,description,gender,age
0,95591,https://s4.anilist.co/file/anilistcdn/staff/la...,Kenjirou Tsuda,"__Height:__ 170 cm (5'7"")\n__Agency:__ Amuleto...",Male,55
1,95118,https://s4.anilist.co/file/anilistcdn/staff/la...,Hiroshi Kamiya,"**Height:** 167 cm (5'6"")\n\n[Agency Profile (...",Male,51
2,95869,https://s4.anilist.co/file/anilistcdn/staff/la...,Saori Hayami,__Agency:__ I'm Enterprise\n\n[Profile](https:...,Female,35
3,95185,https://s4.anilist.co/file/anilistcdn/staff/la...,Kana Hanazawa,"__Height:__ 156 cm (5'1"")\n\n[Profile](https:/...",Female,37
4,95065,https://s4.anilist.co/file/anilistcdn/staff/la...,Mamoru Miyano,"__Height:__ 182 cm (6'0"")\n\n[Website](https:/...",Male,43
...,...,...,...,...,...,...
145,116971,https://s4.anilist.co/file/anilistcdn/staff/la...,Daiki Yamashita,[Profile (Arts Vision)](https://www.artsvision...,Male,37
146,133507,https://s4.anilist.co/file/anilistcdn/staff/la...,Chiaki Kobayashi,**Agency:** Office Osawa\n\n[Twitter](https://...,Male,32
147,119976,https://s4.anilist.co/file/anilistcdn/staff/la...,MYTH & ROID,MYTH & ROID is a musical unit consisting of mu...,NaN,<NA>
148,119416,https://s4.anilist.co/file/anilistcdn/staff/la...,Kaede Hondo,__Agency:__ I'm Enterprise\n\n[Agency Profile]...,Female,30


In [561]:
dfs['user']

,id,name,avatar,createdAt
0,219847,Mand3012,https://s4.anilist.co/file/anilistcdn/user/ava...,2018-12-29 20:33:13
1,61155,7elou,https://s4.anilist.co/file/anilistcdn/user/ava...,2015-12-28 10:10:56
2,63191,killkill,https://s4.anilist.co/file/anilistcdn/user/ava...,2016-01-21 11:46:01
3,76074,MaherAnime,https://s4.anilist.co/file/anilistcdn/user/ava...,2016-07-17 00:49:56
4,34111,Ichigokun,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-11-11 13:47:18
...,...,...,...,...
75,35971,ANON192,https://s4.anilist.co/file/anilistcdn/user/ava...,2014-12-13 08:52:45
76,37820,BlackOcean,https://s4.anilist.co/file/anilistcdn/user/ava...,2015-01-05 05:32:55
77,48303,Enndo,https://s4.anilist.co/file/anilistcdn/user/ava...,2015-06-24 05:14:19
78,76717,qubiq,https://s4.anilist.co/file/anilistcdn/user/ava...,2016-07-29 22:52:13


In [563]:
dfs['genre']

,id_midia,name
0,154587,Adventure
1,154587,Drama
2,154587,Fantasy
3,114129,Action
4,114129,Comedy
...,...,...
1847,30704,Sci-Fi
1848,30869,Action
1849,30869,Adventure
1850,30869,Comedy


In [564]:
dfs['occupation']

,id_staff,name
0,95591,Voice Actor
1,95118,Voice Actor
2,95869,Voice Actor
3,95869,Vocalist
4,95185,Voice Actor
...,...,...
196,133507,Voice Actor
197,119976,Band
198,119416,Voice Actor
199,95027,Voice Actor
